# Interactive neural entrainment explorer

Use the controls below to choose a rhythmic stimulus, run the existing Brian2 excitatory/inhibitory network, and inspect both the input and the resulting network dynamics.

The stimulus is an 8-step repeating pattern. A `1` is an accent/onset and a `0` is silence. The plots show the stimulus rate and onset pattern, a spike raster with the stimulus rate overlaid, population firing rates, a sliding-window PLV time course, representative membrane potentials, polar spike-phase histograms, and E-E/I-I/E-I cross-correlograms. Together these reveal *when* entrainment emerges and *how* spiking is structured around the stimulus cycle, rather than only its average strength.

The default population sizes are intentionally modest so that the notebook stays interactive. Increase them when you want a higher-fidelity run.

In [11]:
# Bookkeeping
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# Make the notebook work when launched from either the repo notebook directory
_here = Path.cwd().resolve()
_project = next((p for p in (_here, _here.parent) if (p / "src" / "model.py").exists()), None)
if _project is None:
    raise FileNotFoundError("Could not find auditory-entrainment/src/model.py")
if str(_project) not in sys.path:
    sys.path.insert(0, str(_project))

# Reload project modules so edits to src/*.py take effect on re-run.
import importlib
for _name in ("src.model", "src.stimuli", "experiments.measures.synchrony"):
    if _name in sys.modules:
        importlib.reload(sys.modules[_name])

from src.model import Params, simulate
from src.stimuli import PATTERNS, PATTERN_NAMES, Rhythm, make_rhythm
from experiments.measures import synchrony

plt.rcParams.update({"figure.figsize": (12, 9), "axes.grid": True, "grid.alpha": 0.25})
print(f"Using project: {_project}")

Using project: C:\Users\local\Desktop\neural_entrainment_synchrony\auditory-entrainment


## Choose a stimulus and network settings

Select a preset or enter your own 8-bit pattern such as `1 0 0 1 0 1 0 0`. The **Run simulation** button is deliberately explicit because each change rebuilds the Brian2 network.

In [12]:
_pattern_options = [(f"{pid}: {PATTERN_NAMES[pid]}  {PATTERNS[pid]}", pid) for pid in PATTERNS]
preset = widgets.Dropdown(options=_pattern_options, value=2, description="Preset:")
custom_pattern = widgets.Text(value="", description="Custom 8-bit:", placeholder="e.g. 1 0 0 1 0 1 0 0")
field_layout = widgets.Layout(width="145px")
field_style = {"description_width": "62px"}
duration = widgets.BoundedIntText(value=4000, min=1000, max=60000, step=200, description="dur (ms)", layout=field_layout, style=field_style)
step_duration = widgets.BoundedFloatText(value=0.25, min=0.05, max=1.0, step=0.05, description="step (s)", layout=field_layout, style=field_style)
rate_on = widgets.BoundedFloatText(value=35.0, min=5.0, max=100.0, step=5.0, description="on (Hz)", layout=field_layout, style=field_style)
rate_off = widgets.BoundedFloatText(value=0.0, min=0.0, max=10.0, step=1.0, description="off (Hz)", layout=field_layout, style=field_style)
jitter_ms = widgets.BoundedFloatText(value=0.0, min=0.0, max=50.0, step=1.0, description="jitter (ms)", layout=field_layout, style=field_style)
i_bias = widgets.BoundedFloatText(value=100.0, min=0.0, max=180.0, step=5.0, description="bias (pA)", layout=field_layout, style=field_style)
i_bias_std = widgets.BoundedFloatText(value=0.0, min=0.0, max=60.0, step=2.5, description="bias SD", layout=field_layout, style=field_style)
weight_sigma = widgets.BoundedFloatText(value=0.0, min=0.0, max=1.5, step=0.05, description="w log SD", layout=field_layout, style=field_style)
w_thal = widgets.BoundedFloatText(value=0.05, min=0.0, max=0.2, step=0.01, description="stim w", layout=field_layout, style=field_style)
w_ee = widgets.BoundedFloatText(value=1.5, min=0.0, max=10.0, step=0.1, description="w EE", layout=field_layout, style=field_style)
w_ei = widgets.BoundedFloatText(value=1.0, min=0.0, max=10.0, step=0.1, description="w EI", layout=field_layout, style=field_style)
w_ie = widgets.BoundedFloatText(value=3.0, min=0.0, max=10.0, step=0.1, description="w IE", layout=field_layout, style=field_style)
w_ii = widgets.BoundedFloatText(value=4.0, min=0.0, max=10.0, step=0.1, description="w II", layout=field_layout, style=field_style)
p_ee = widgets.BoundedFloatText(value=0.1, min=0.0, max=1.0, step=0.05, description="p EE", layout=field_layout, style=field_style)
p_ei = widgets.BoundedFloatText(value=0.4, min=0.0, max=1.0, step=0.05, description="p EI", layout=field_layout, style=field_style)
p_ie = widgets.BoundedFloatText(value=0.5, min=0.0, max=1.0, step=0.05, description="p IE", layout=field_layout, style=field_style)
p_ii = widgets.BoundedFloatText(value=0.5, min=0.0, max=1.0, step=0.05, description="p II", layout=field_layout, style=field_style)
p_thal = widgets.BoundedFloatText(value=0.5, min=0.0, max=1.0, step=0.05, description="p thal", layout=field_layout, style=field_style)
delay_ei = widgets.BoundedFloatText(value=1.0, min=0.0, max=10.0, step=0.1, description="E-I (ms)", layout=field_layout, style=field_style)
delay_ie = widgets.BoundedFloatText(value=1.0, min=0.0, max=10.0, step=0.1, description="I-E (ms)", layout=field_layout, style=field_style)
delay_ei_std = widgets.BoundedFloatText(value=0.0, min=0.0, max=10.0, step=0.1, description="E-I SD", layout=field_layout, style=field_style)
delay_ie_std = widgets.BoundedFloatText(value=0.0, min=0.0, max=10.0, step=0.1, description="I-E SD", layout=field_layout, style=field_style)
noise = widgets.BoundedFloatText(value=0.0, min=0.0, max=8.0, step=0.5, description="noise (mV)", layout=field_layout, style=field_style)
n_exc = widgets.BoundedIntText(value=120, min=20, max=4000, step=20, description="N exc", layout=field_layout, style=field_style)
n_inh = widgets.BoundedIntText(value=40, min=10, max=1200, step=10, description="N inh", layout=field_layout, style=field_style)
seed = widgets.IntText(value=42, description="seed", layout=field_layout, style=field_style)
show_background = widgets.Checkbox(value=False, description="Add on-beat background")
run_button = widgets.Button(description="Run simulation", button_style="success", icon="play")
output = widgets.Output()
parameter_name = widgets.Text(value="", description="set name", placeholder="e.g. gamma baseline")
saved_sets = widgets.Dropdown(options=[("(none)", "")], description="load set")
save_parameters_button = widgets.Button(description="Save parameters", button_style="info", icon="save")
load_parameters_button = widgets.Button(description="Load parameters", button_style="", icon="download")
parameter_status = widgets.Output()

controls = widgets.VBox([
    widgets.HBox([preset, custom_pattern]),
    widgets.HBox([duration, step_duration, rate_on, rate_off, jitter_ms]),
    widgets.HBox([i_bias, i_bias_std, weight_sigma, w_thal, noise]),
    widgets.HBox([w_ee, w_ei, w_ie, w_ii]),
    widgets.HBox([p_ee, p_ei, p_ie, p_ii, p_thal]),
    widgets.HBox([delay_ei, delay_ie, delay_ei_std, delay_ie_std]),
    widgets.HBox([n_exc, n_inh, seed, show_background]),
    widgets.HBox([parameter_name, save_parameters_button, saved_sets, load_parameters_button]),
    run_button,
])
display(controls, parameter_status, output)

Output()

Output()

In [13]:
def _read_pattern():
        """
        Returns the selected pattern
        Validates custom input
        """

        text = custom_pattern.value.strip()
        if text:
            tokens = text.replace(",", " ").split()
            try:
                pattern = tuple(int(token) for token in tokens)
            except ValueError as exc:
                raise ValueError("Custom pattern must contain only 0 and 1 values.") from exc
            if len(pattern) != 8 or any(bit not in (0, 1) for bit in pattern):
                raise ValueError("Custom pattern must contain exactly eight 0/1 values.")
            return pattern, "custom"
        return tuple(PATTERNS[preset.value]), PATTERN_NAMES[preset.value]


def _time_resolved_plv(rate, fs, freq, window_s=0.1, step_s=0.05):
    """
    Windowed PLV between a population rate and stimulus frequnecy
    """

    rate = np.asarray(rate, dtype=float)
    win = int(round(window_s * fs))
    step = max(1, int(round(step_s * fs)))
    if win < 4 or rate.size < win:
        return np.empty(0), np.empty(0)
    starts = np.arange(0, rate.size - win + 1, step)
    times = (starts + win / 2.0) / fs
    plvs = np.array([synchrony.plv(rate[s:s + win], fs, freq) for s in starts])
    return times, plvs


def _bin_trains(spike_times_s, spike_indices, n_units, duration_s, bin_s):
    """
    Returns an (n_units, n_bins) matrix of mean subtracted spike counts
    """

    n_units = int(n_units)
    n_bins = max(1, int(np.ceil(duration_s / bin_s)))
    spike_times_s = np.asarray(spike_times_s, dtype=float)
    spike_indices = np.asarray(spike_indices, dtype=np.int64)
    counts = np.zeros((n_units, n_bins), dtype=float)
    if spike_times_s.size and n_units > 0:
        keep = (spike_indices >= 0) & (spike_indices < n_units)
        times = spike_times_s[keep]
        idx = spike_indices[keep]
        bins = np.clip((times / bin_s).astype(np.int64), 0, n_bins - 1)
        flat = np.bincount(idx * n_bins + bins, minlength=n_units * n_bins)
        counts = flat.reshape(n_units, n_bins).astype(float)
    return counts - counts.mean(axis=1, keepdims=True)


def _average_xcorr(trains_a, trains_b=None, bin_ms=1.0, max_lag_ms=50.0):
    """
    Average cross corellogram spike trains
    By default trains_b averages within population
    """

    bin_s = bin_ms / 1000.0
    max_lag_bins = int(round(max_lag_ms / bin_ms))
    lags = np.arange(-max_lag_bins, max_lag_bins + 1) * bin_ms
    a = np.asarray(trains_a, dtype=float)
    b = None if trains_b is None else np.asarray(trains_b, dtype=float)
    if a.ndim != 2 or a.shape[0] < 1 or a.shape[1] == 0:
        return lags, np.zeros(lags.size)
    n_bins = a.shape[1]
    if b is None:
        if a.shape[0] < 2:
            return lags, np.zeros(lags.size)
        n_units = a.shape[0]
        acc = np.correlate(a.sum(axis=0), a.sum(axis=0), mode="full")
        for i in range(n_units):
            acc -= np.correlate(a[i], a[i], mode="full")
        acc /= n_units * (n_units - 1)
        norm = n_bins * float(a.std(axis=1).mean()) ** 2
    else:
        if b.ndim != 2 or b.shape[0] < 1 or b.shape[1] != n_bins:
            return lags, np.zeros(lags.size)
        acc = np.correlate(a.sum(axis=0), b.sum(axis=0), mode="full") / (a.shape[0] * b.shape[0])
        norm = n_bins * float(a.std(axis=1).mean()) * float(b.std(axis=1).mean())
    if norm > 0:
        acc = acc / norm
    mid = acc.size // 2
    return lags, acc[mid - max_lag_bins: mid + max_lag_bins + 1]


def _phase_histogram(ax, spike_times_s, freq, color, title):
    """
    Polar histogram of spike phases relative to the stimulus cycle
    """

    phases = (np.asarray(spike_times_s, dtype=float) * freq) % 1.0
    angles = phases * 2.0 * np.pi

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(title, pad=16)

    # draw invisible bar when there are no spikes
    if phases.size == 0:
        ax.bar([0.0], [0.0], width=0.1, color=color, alpha=0.0)
        ax.text(0.5, -0.2, "no spikes", transform=ax.transAxes,
                ha="center", va="top", fontsize=9)
        return
    
    ax.hist(angles, bins=24, color=color, alpha=0.75, density=True)
    
    z = np.mean(np.exp(1j * angles))
    mean_phase = (np.angle(z) / (2.0 * np.pi)) % 1.0
    ax.annotate("", xy=(np.angle(z), np.abs(z)), xytext=(0.0, 0.0),
                arrowprops=dict(arrowstyle="->", color="black", lw=2.0))
    ax.text(0.5, -0.2, f"VS={np.abs(z):.3f} | mean phase={mean_phase:.2f} cyc",
            transform=ax.transAxes, ha="center", va="top", fontsize=9)


def _plot_result(result, rhythm, label, background=None):
    duration_s = result.params.duration / 1000
    fs = 1.0 / float(np.mean(np.diff(result.rate_t)))
    freq = rhythm.pulse_frequency
    t_stim = np.linspace(0, duration_s, 2000, endpoint=False)
    stimulus_rate = rhythm.rate(t_stim)
    n_pattern_steps = int(np.ceil(duration_s / rhythm.step_duration))
    pattern_values = np.tile(rhythm.pattern, n_pattern_steps + 1)[:n_pattern_steps + 1]
    pattern_times = np.arange(pattern_values.size) * rhythm.step_duration * 1000
    t_rate = result.rate_t * 1000
    t_voltage = result.v_t * 1000
    exc_mask = result.spike_indices < result.n_exc
    inh_mask = ~exc_mask
    exc_spike_t = result.spike_times[exc_mask] * 1000
    inh_spike_t = result.spike_times[inh_mask] * 1000
    exc_spike_s = result.spike_times[exc_mask]
    inh_spike_s = result.spike_times[inh_mask]
    exc_idx = result.spike_indices[exc_mask]
    inh_idx = result.spike_indices[inh_mask] - result.n_exc

    fig = plt.figure(figsize=(14, 22))
    gs = fig.add_gridspec(7, 1, height_ratios=[1.0, 1.7, 1.2, 1.1, 1.5, 1.7, 1.4], hspace=0.6)

    ax = fig.add_subplot(gs[0])
    ax.plot(t_stim * 1000, stimulus_rate, color="tab:orange", lw=1.5, label="selected stimulus")
    if background is not None:
        ax.plot(t_stim * 1000, background.rate(t_stim), color="tab:green", lw=1.3, linestyle="--", label="on-beat background")
    ax.step(pattern_times, pattern_values, where="post", color="black", alpha=0.7, label="onset pattern")
    ax.set_ylabel("Input (Hz) / onset")
    ax.set_title(f"Stimulus: {label} | cycle {rhythm.cycle_duration:.2f} s | fundamental {rhythm.pulse_frequency:.2f} Hz")
    ax.legend(loc="upper right")

    ax = fig.add_subplot(gs[1])
    if exc_spike_t.size:
        ax.scatter(exc_spike_t, result.spike_indices[exc_mask], s=3, color="tab:blue", label="excitatory")
    if inh_spike_t.size:
        ax.scatter(inh_spike_t, result.spike_indices[inh_mask] - result.n_exc, s=3, color="tab:red", label="inhibitory")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Neuron index")
    ax.set_title("Spike raster with stimulus rate overlay")

    if exc_spike_t.size or inh_spike_t.size:
        ax.legend(loc="upper right", markerscale=3)
    ax_stim = ax.twinx()
    ax_stim.plot(t_stim * 1000, stimulus_rate, color="tab:orange", alpha=0.3, lw=1.0)
    ax_stim.set_ylabel("Stimulus (Hz)", color="tab:orange")
    ax_stim.tick_params(axis="y", colors="tab:orange")
    ax_stim.set_ylim(bottom=0)
    ax_stim.grid(False)

    ax = fig.add_subplot(gs[2])
    ax.plot(t_rate, result.rate_exc, color="tab:blue", label="excitatory rate")
    ax.plot(t_rate, result.rate_inh, color="tab:red", label="inhibitory rate")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Rate (Hz)")
    ax.set_title("Population activity")
    ax.legend(loc="upper right")

    window_s = 0.5
    t_plv_e, plv_e_tc = _time_resolved_plv(result.rate_exc, fs, freq, window_s=window_s)
    t_plv_i, plv_i_tc = _time_resolved_plv(result.rate_inh, fs, freq, window_s=window_s)
    ax = fig.add_subplot(gs[3])
    if t_plv_e.size:
        ax.plot(t_plv_e * 1000, plv_e_tc, color="tab:blue", lw=1.5, label="excitatory")
    if t_plv_i.size:
        ax.plot(t_plv_i * 1000, plv_i_tc, color="tab:red", lw=1.5, label="inhibitory")
    ax.axhline(0.0, color="grey", lw=0.5)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("PLV")
    ax.set_title(f"Time-resolved PLV to {freq:.2f} Hz (window {window_s * 1000:.0f} ms)")
    ax.legend(loc="upper right")

    ax = fig.add_subplot(gs[4])
    n_trace = min(8, result.n_exc)
    for trace in result.v_exc[:n_trace]:
        ax.plot(t_voltage, trace, color="tab:blue", alpha=0.18, lw=0.7)
    mean_v = result.v_exc.mean(axis=0)
    ax.plot(t_voltage, mean_v, color="black", lw=1.8, label="mean excitatory voltage")
    ax.plot(t_voltage, result.v_inh.mean(axis=0), color="tab:red", lw=1.3, label="mean inhibitory voltage")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Voltage (mV)")
    ax.set_title("Network dynamics: membrane-potential traces and population means")
    ax.legend(loc="upper right")

    sub_polar = gs[5].subgridspec(1, 2, wspace=0.35)
    _phase_histogram(fig.add_subplot(sub_polar[0], projection="polar"), exc_spike_s, freq, "tab:blue", "Excitatory spike phase")
    _phase_histogram(fig.add_subplot(sub_polar[1], projection="polar"), inh_spike_s, freq, "tab:red", "Inhibitory spike phase")

    bin_ms, max_lag_ms = 1.0, 50.0
    bin_s = bin_ms / 1000.0
    trains_e = _bin_trains(exc_spike_s, exc_idx, result.n_exc, duration_s, bin_s)
    trains_i = _bin_trains(inh_spike_s, inh_idx, result.n_inh, duration_s, bin_s)
    lags, xcorr_ee = _average_xcorr(trains_e, bin_ms=bin_ms, max_lag_ms=max_lag_ms)
    _, xcorr_ii = _average_xcorr(trains_i, bin_ms=bin_ms, max_lag_ms=max_lag_ms)
    _, xcorr_ei = _average_xcorr(trains_e, trains_i, bin_ms=bin_ms, max_lag_ms=max_lag_ms)
    sub_x = gs[6].subgridspec(1, 3, wspace=0.3)
    for pos, values, title, color in (
        (0, xcorr_ee, "E-E", "tab:blue"),
        (1, xcorr_ii, "I-I", "tab:red"),
        (2, xcorr_ei, "E-I", "tab:purple"),
    ):
        axis = fig.add_subplot(sub_x[pos])
        axis.axhline(0.0, color="grey", lw=0.6)
        axis.bar(lags, values, width=bin_ms * 0.9, color=color, alpha=0.85)
        axis.set_xlabel("Lag (ms)")
        axis.set_ylabel("Correlation")
        axis.set_title(f"{title} cross-correlogram")
    # fig.tight_layout()
    display(fig)
    plt.close(fig)

    mean_exc = float(result.rate_exc.mean())
    mean_inh = float(result.rate_inh.mean())
    plv_exc = synchrony.plv(result.rate_exc, fs, freq)
    plv_inh = synchrony.plv(result.rate_inh, fs, freq)
    vs_exc = synchrony.vector_strength(exc_spike_s, freq)
    vs_inh = synchrony.vector_strength(inh_spike_s, freq)
    corr_exc = synchrony.spike_pairwise_corr(exc_spike_s, exc_idx, result.n_exc, duration_s)
    corr_inh = synchrony.spike_pairwise_corr(inh_spike_s, inh_idx, result.n_inh, duration_s)
    print(f"Mean rates: excitatory {mean_exc:.2f} Hz | inhibitory {mean_inh:.2f} Hz")
    print(f"PLV to {freq:.2f} Hz:  E {plv_exc:.3f} | I {plv_inh:.3f}")
    print(f"Vector strength to {freq:.2f} Hz:  E {vs_exc:.3f} | I {vs_inh:.3f}")
    print(f"Spike-count correlation:  E {corr_exc:.3f} | I {corr_inh:.3f}")

PARAMETER_FILE = _project / "notebooks" / "parameter_sets.json"

_PARAMETER_WIDGETS = {
    "duration": duration,
    "step_duration": step_duration,
    "rate_on": rate_on,
    "rate_off": rate_off,
    "jitter_ms": jitter_ms,
    "I_bias": i_bias,
    "I_bias_std": i_bias_std,
    "weight_sigma": weight_sigma,
    "w_thal": w_thal,
    "w_ee": w_ee,
    "w_ei": w_ei,
    "w_ie": w_ie,
    "w_ii": w_ii,
    "p_ee": p_ee,
    "p_ei": p_ei,
    "p_ie": p_ie,
    "p_ii": p_ii,
    "p_thal": p_thal,
    "delay_ei": delay_ei,
    "delay_ie": delay_ie,
    "delay_ei_std": delay_ei_std,
    "delay_ie_std": delay_ie_std,
    "noise": noise,
    "n_exc": n_exc,
    "n_inh": n_inh,
    "seed": seed,
    "show_background": show_background,
}


def _parameter_values():
    return {
        "preset": preset.value,
        "custom_pattern": custom_pattern.value,
        **{key: widget.value for key, widget in _PARAMETER_WIDGETS.items()},
    }


def _refresh_saved_sets():
    if PARAMETER_FILE.exists():
        with PARAMETER_FILE.open(encoding="utf-8") as handle:
            saved = json.load(handle)
        saved_sets.options = [(name, name) for name in sorted(saved)] or [("(none)", "")]
    else:
        saved_sets.options = [("(none)", "")]


def save_parameters(_button=None):
    name = parameter_name.value.strip()
    with parameter_status:
        clear_output(wait=True)
        if not name:
            print("Enter a parameter-set name before saving.")
            return
        try:
            saved = {}
            if PARAMETER_FILE.exists():
                with PARAMETER_FILE.open(encoding="utf-8") as handle:
                    saved = json.load(handle)
            saved[name] = _parameter_values()
            PARAMETER_FILE.parent.mkdir(parents=True, exist_ok=True)
            with PARAMETER_FILE.open("w", encoding="utf-8") as handle:
                json.dump(saved, handle, indent=2)
            _refresh_saved_sets()
            print(f"Saved '{name}' to {PARAMETER_FILE.name}")
        except (OSError, TypeError, ValueError, json.JSONDecodeError) as exc:
            print(f"Could not save parameters: {exc}")


def load_parameters(_button=None):
    name = saved_sets.value
    with parameter_status:
        clear_output(wait=True)
        if not name:
            print("Select a saved parameter set first.")
            return
        try:
            with PARAMETER_FILE.open(encoding="utf-8") as handle:
                values = json.load(handle)[name]
            preset.value = values["preset"]
            custom_pattern.value = values["custom_pattern"]
            for key, widget in _PARAMETER_WIDGETS.items():
                widget.value = values[key]
            print(f"Loaded '{name}'")
        except (KeyError, OSError, TypeError, ValueError, json.JSONDecodeError) as exc:
            print(f"Could not load parameters: {exc}")


_refresh_saved_sets()
save_parameters_button.on_click(save_parameters)
load_parameters_button.on_click(load_parameters)


def run_simulation(_button=None):
    with output:
        clear_output(wait=True)
        try:
            pattern, label = _read_pattern()
            rhythm = Rhythm(
                pattern,
                step_duration=step_duration.value,
                rate_on=rate_on.value,
                rate_off=rate_off.value,
                jitter_ms=jitter_ms.value,
                name=label,
            )
            background = (
                Rhythm(
                    (1, 0, 1, 0, 1, 0, 1, 0),
                    step_duration=step_duration.value,
                    rate_on=rate_on.value,
                    jitter_ms=jitter_ms.value,
                    name="on-beat background",
                )
                if show_background.value
                else None
            )
            params = Params(
                duration=duration.value,
                dt=0.1,
                v_dt=1.0,
                seed=seed.value,
                n_exc=n_exc.value,
                n_inh=n_inh.value,
                n_thal=30,
                w_thal=w_thal.value,
                p_thal=p_thal.value,
                w_ee=w_ee.value,
                w_ei=w_ei.value,
                w_ie=w_ie.value,
                w_ii=w_ii.value,
                p_ee=p_ee.value,
                p_ei=p_ei.value,
                p_ie=p_ie.value,
                p_ii=p_ii.value,
                delay_ei=delay_ei.value,
                delay_ie=delay_ie.value,
                delay_ei_std=delay_ei_std.value,
                delay_ie_std=delay_ie_std.value,
                bg_noise=noise.value,
                I_bias=i_bias.value,
                I_bias_std=i_bias_std.value,
                weight_lognormal_sigma=weight_sigma.value,
            )


            t0 = time.perf_counter()

            print("Running Brian2 simulation...")
            result = simulate(params, stimulus=rhythm, background=background)
            t1 = time.perf_counter()
            print(f"  simulation: {t1 - t0:.1f} s")

            _plot_result(result, rhythm, label, background=background)
            t2 = time.perf_counter()
            print(f"  plot + PLV: {t2 - t1:.1f} s")
            print(f"  total:      {t2 - t0:.1f} s")

            print(result.params.describe())
            

        except Exception as exc:
            print(f"Could not run simulation: {exc}")


run_button.on_click(run_simulation)
run_simulation()
